## Verifica deep models

### SETUP fair pipeline

In [1]:
from deep_models import *
from data_pipeline import *
import random
import numpy as np
import tensorflow as tf

#split and clean
df = load_data('data/data-ready-def.xlsx')
train_df, valid_df, test_df = split_data(df)  #i tre set splittati

train_df_clean = clean_data(train_df)
valid_df_clean = clean_data(valid_df)
test_df_clean = clean_data(test_df)

#normalize
train_df_norm, valid_df_norm, test_df_norm, train_df_min, train_df_max = normalize(train_df_clean, valid_df_clean, test_df_clean)

#create_sequence
train_X, train_y = create_sequences(train_df_norm, 20, 1)
valid_X, valid_y = create_sequences(valid_df_norm, 20, 1)
test_X, test_y = create_sequences(test_df_norm, 20, 1)

print(train_X.shape, train_y.shape)
print(valid_X.shape, valid_y.shape)
print(test_X.shape, test_y.shape)

(2261, 20, 8) (2261,)
(399, 20, 8) (399,)
(399, 20, 8) (399,)


### LSTM

In [ ]:
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

m = build_lstm_model(train_X.shape[2], train_X.shape[1], lstm_units=64)

lstm_valid_mae, lstm_valid_rmse, lstm_valid_r2, lstm_test_mae, lstm_test_rmse, lstm_test_r2, history = fit_and_evaluate_deep(
    m, train_X, train_y, valid_X, valid_y, test_X, test_y,
    train_df_min['Amount of irrigation'], train_df_max['Amount of irrigation']
)

print(lstm_valid_mae, lstm_valid_rmse, lstm_valid_r2)
print(lstm_test_mae, lstm_test_rmse, lstm_test_r2)

#verifica valori corretti
ufficiali_lstm = (1.686422, 2.065487, 0.166424, 1.613117, 1.908872, 0.255162)
ottenuti_lstm = (lstm_valid_mae, lstm_valid_rmse, lstm_valid_r2, lstm_test_mae, lstm_test_rmse, lstm_test_r2)
assert np.allclose(ottenuti_lstm, ufficiali_lstm, atol=1e-5), f"Mismatch LSTM: {ottenuti_lstm} vs {ufficiali_lstm}"
print("LSTM: OK, corrisponde ai numeri ufficiali")

### CNN

In [ ]:
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

m = build_cnn_model(train_X.shape[2], train_X.shape[1], filters=64)

cnn_valid_mae, cnn_valid_rmse, cnn_valid_r2, cnn_test_mae, cnn_test_rmse, cnn_test_r2, history = fit_and_evaluate_deep(
    m, train_X, train_y, valid_X, valid_y, test_X, test_y,
    train_df_min['Amount of irrigation'], train_df_max['Amount of irrigation']
)

print(cnn_valid_mae, cnn_valid_rmse, cnn_valid_r2)
print(cnn_test_mae, cnn_test_rmse, cnn_test_r2)

ufficiali_cnn = (1.689530, 2.119448, 0.122300, 1.629626, 1.969049, 0.207461)
ottenuti_cnn = (cnn_valid_mae, cnn_valid_rmse, cnn_valid_r2, cnn_test_mae, cnn_test_rmse, cnn_test_r2)
assert np.allclose(ottenuti_cnn, ufficiali_cnn, atol=1e-5), f"Mismatch CNN: {ottenuti_cnn} vs {ufficiali_cnn}"
print("CNN: OK, corrisponde ai numeri ufficiali")

### BiLSTM-CNN-Attention

In [ ]:
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

m = build_bilstm_cnn_attention_model(train_X.shape[2], train_X.shape[1], lstm_units=64)

bilstm_cnn_attn_valid_mae, bilstm_cnn_attn_valid_rmse, bilstm_cnn_attn_valid_r2, bilstm_cnn_attn_test_mae, bilstm_cnn_attn_test_rmse, bilstm_cnn_attn_test_r2, history = fit_and_evaluate_deep(
    m, train_X, train_y, valid_X, valid_y, test_X, test_y,
    train_df_min['Amount of irrigation'], train_df_max['Amount of irrigation']
)

print(bilstm_cnn_attn_valid_mae, bilstm_cnn_attn_valid_rmse, bilstm_cnn_attn_valid_r2)
print(bilstm_cnn_attn_test_mae, bilstm_cnn_attn_test_rmse, bilstm_cnn_attn_test_r2)

ufficiali_bilstm_cnn_attn = (1.764591846719496, 2.126648124995951, 0.11632657142119429, 1.6902393322131837, 1.978061936382966, 0.20018855974647864)
ottenuti_bilstm_cnn_attn = (bilstm_cnn_attn_valid_mae, bilstm_cnn_attn_valid_rmse, bilstm_cnn_attn_valid_r2, bilstm_cnn_attn_test_mae, bilstm_cnn_attn_test_rmse, bilstm_cnn_attn_test_r2)
assert np.allclose(ottenuti_bilstm_cnn_attn, ufficiali_bilstm_cnn_attn, atol=1e-9), f"Mismatch BiLSTM-CNN-Attention: {ottenuti_bilstm_cnn_attn} vs {ufficiali_bilstm_cnn_attn}"
print("BiLSTM-CNN-Attention: OK, corrisponde ai numeri ufficiali")

### Risultati

In [ ]:
import pandas as pd

results_deep = pd.DataFrame([
    {'model': 'LSTM', 'mae_valid': lstm_valid_mae, 'rmse_valid': lstm_valid_rmse, 'r2_valid': lstm_valid_r2,
     'mae_test': lstm_test_mae, 'rmse_test': lstm_test_rmse, 'r2_test': lstm_test_r2},
    {'model': 'CNN', 'mae_valid': cnn_valid_mae, 'rmse_valid': cnn_valid_rmse, 'r2_valid': cnn_valid_r2,
     'mae_test': cnn_test_mae, 'rmse_test': cnn_test_rmse, 'r2_test': cnn_test_r2},
    {'model': 'BiLSTM-CNN-Attention', 'mae_valid': bilstm_cnn_attn_valid_mae, 'rmse_valid': bilstm_cnn_attn_valid_rmse, 'r2_valid': bilstm_cnn_attn_valid_r2,
     'mae_test': bilstm_cnn_attn_test_mae, 'rmse_test': bilstm_cnn_attn_test_rmse, 'r2_test': bilstm_cnn_attn_test_r2},
])

results_deep.to_csv('results/deep_models_H1.csv', index=False)
results_deep